# Foundry IQ Cookbook — Build a Grounded Knowledge Base

Modern agents need more than vector search — they need retrieval that **reasons over the question**, runs **parallel subqueries**, **reranks** with a semantic ranker, and **synthesizes** an answer with citations.

**Foundry IQ** is Microsoft's intelligence layer for agentic retrieval, built on Azure AI Search. This cookbook walks you through the full pipeline end-to-end using the Azure AI Search Python SDK (`12.1.0a*` alpha — the `2026-05-01-preview` API surface that backs the upcoming Build 2026 release).

By the end of this notebook you will have:

1. Provisioned a **Search Index** with vector + semantic configuration
2. Indexed the NASA *Earth at Night* dataset (no separate embedding pipeline — the indexer vectorizes for you)
3. Created a **Knowledge Source** pointing at the index
4. Created a **Knowledge Base** that pairs the source with a chat model and answer synthesis
5. Run a **complex multi-part query** and inspected the planner's subqueries and citations
6. Continued the conversation with a **multi-turn follow-up** that preserves context
7. Talked to the Knowledge Base **directly over MCP** (JSON-RPC `tools/call`)
8. Plugged the Knowledge Base into a **Microsoft Agent Framework** agent via `MCPStreamableHTTPTool`
9. Plugged the Knowledge Base into the **Foundry Agent Service** via a `RemoteTool` project connection
7. Talked to the Knowledge Base **directly over MCP** (JSON-RPC `tools/call`)
8. Plugged the Knowledge Base into a **Microsoft Agent Framework** agent via `MCPStreamableHTTPTool`
9. Plugged the Knowledge Base into the **Foundry Agent Service** via a `RemoteTool` project connection
10. **Cleaned up** every resource the notebook created

## Architecture

```text
            ┌──────────────────────────────────────────────────────┐
            │                  Foundry IQ Pipeline                 │
            └──────────────────────────────────────────────────────┘
   query →  Knowledge Base ──► LLM planner ──► parallel subqueries ─┐
                │                                                   ▼
                │                                          Knowledge Source
                │                                                   │
                ▼                                                   ▼
        Answer synthesis ◄── reranked results ◄── Search Index (vector + semantic)
                │
                ▼
        Cited answer + activity trace
```


## Prerequisites

- An **Azure AI Search** service in a [region that supports agentic retrieval](https://learn.microsoft.com/azure/search/search-region-support).
- A **Microsoft Foundry** (or Azure OpenAI) resource with two deployments:
  - An embedding model — `text-embedding-3-large` recommended (3072 dimensions, used in this notebook).
  - A chat completion model — `gpt-4o`, `gpt-4o-mini`, or `gpt-5-mini` all work.
- Admin keys for both services (this notebook uses key auth for portability; production deployments should prefer managed identity — see the [Foundry IQ private setup guide](https://learn.microsoft.com/azure/search/search-security-rbac)).

### Configure your environment

Copy `.env.example` to `.env` in this folder and fill in your endpoints and keys:

```env
SEARCH_ENDPOINT=https://<your-search-service>.search.windows.net
SEARCH_API_KEY=<your-search-admin-key>
AOAI_ENDPOINT=https://<your-foundry-resource>.openai.azure.com
AOAI_API_KEY=<your-azure-openai-key>
```

The `.env.example` file lists every variable and its default.

### Install dependencies

The alpha SDK lives on the Azure SDK public feed. The accompanying `pip.ini` and `requirements.txt` in this folder pin the exact versions used here. The cell below installs them and silences pip output for a clean notebook.

In [1]:
%%capture
%pip install -r requirements.txt --extra-index-url https://pkgs.dev.azure.com/azure-sdk/public/_packaging/azure-sdk-for-python/pypi/simple/

## Step 1 — Configure the client

Load the `.env`, resolve sensible defaults for optional values, and build a single `AzureKeyCredential` and `SearchIndexClient` that the rest of the notebook reuses.

`azure_openai_resource_uri()` trims any `/openai/...` path off the endpoint — handy if your `.env` already contains a full deployment URL.

In [2]:
import os
from pathlib import Path

from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from dotenv import load_dotenv

load_dotenv(override=True)


def env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


def azure_openai_resource_uri(endpoint: str) -> str:
    # Accept either the resource root or a full deployment URL.
    return endpoint.split("/openai/", 1)[0].rstrip("/")


# Service endpoints + keys
SEARCH_ENDPOINT = env("SEARCH_ENDPOINT")
SEARCH_API_KEY = env("SEARCH_API_KEY")
AOAI_ENDPOINT = azure_openai_resource_uri(env("AOAI_ENDPOINT"))
AOAI_API_KEY = env("AOAI_API_KEY")

# Model deployments (override in .env if your deployment names differ)
EMBEDDING_MODEL = os.getenv("AOAI_EMBEDDING_MODEL", "text-embedding-3-large")
EMBEDDING_DEPLOYMENT = os.getenv("AOAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
GPT_MODEL = os.getenv("AOAI_GPT_MODEL", "gpt-4o")
GPT_DEPLOYMENT = os.getenv("AOAI_GPT_DEPLOYMENT", "gpt-4o")

# Resource names — all three are created in this notebook and deleted at the end
INDEX_NAME = os.getenv("INDEX_NAME", "earth-at-night")
KNOWLEDGE_SOURCE_NAME = os.getenv("KNOWLEDGE_SOURCE_NAME", "earth-knowledge-source")
KNOWLEDGE_BASE_NAME = os.getenv("KNOWLEDGE_BASE_NAME", "earth-knowledge-base")

credential = AzureKeyCredential(SEARCH_API_KEY)
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

print(f"Search service : {SEARCH_ENDPOINT}")
print(f"Foundry / AOAI : {AOAI_ENDPOINT}")
print(f"Embeddings     : {EMBEDDING_DEPLOYMENT} ({EMBEDDING_MODEL})")
print(f"Chat model     : {GPT_DEPLOYMENT} ({GPT_MODEL})")

Search service : https://fsunavala-srch-demos-prod.search.windows.net
Foundry / AOAI : https://fsunavala-openai-swecen.openai.azure.com
Embeddings     : text-embedding-3-large (text-embedding-3-large)
Chat model     : gpt-5-mini (gpt-5-mini)


## Step 2 — Create the search index

A Knowledge Source needs an underlying index to search over. We declare four fields:

| Field | Purpose |
|---|---|
| `id` | Document key |
| `page_chunk` | Searchable text content |
| `page_embedding_text_3_large` | 3072-dim vector embedding |
| `page_number` | Filterable page metadata for citations |

Three configurations make this index *agentic-ready*:

- **`vector_search`** — HNSW algorithm plus an `AzureOpenAIVectorizer` so the service can embed query strings on the fly. You never have to embed at query time.
- **`semantic_search`** — required for agentic retrieval. The Knowledge Base uses the semantic ranker to rerank candidates before synthesis.
- A single `default_configuration_name` on the semantic search block so the Knowledge Source can find it without extra plumbing.

In [3]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    HnswAlgorithmConfiguration,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    VectorSearch,
    VectorSearchProfile,
)

fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True,
        sortable=True,
        facetable=True,
    ),
    SearchField(name="page_chunk", type=SearchFieldDataType.String),
    SearchField(
        name="page_embedding_text_3_large",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        vector_search_dimensions=3072,
        vector_search_profile_name="hnsw_text_3_large",
    ),
    SimpleField(
        name="page_number",
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True,
        facetable=True,
    ),
]

embedding_parameters = AzureOpenAIVectorizerParameters(
    resource_url=AOAI_ENDPOINT,
    deployment_name=EMBEDDING_DEPLOYMENT,
    api_key=AOAI_API_KEY,
    model_name=EMBEDDING_MODEL,
)

vector_search = VectorSearch(
    profiles=[
        VectorSearchProfile(
            name="hnsw_text_3_large",
            algorithm_configuration_name="alg",
            vectorizer_name="azure_openai_text_3_large",
        )
    ],
    algorithms=[HnswAlgorithmConfiguration(name="alg")],
    vectorizers=[
        AzureOpenAIVectorizer(
            vectorizer_name="azure_openai_text_3_large",
            parameters=embedding_parameters,
        )
    ],
)

semantic_search = SemanticSearch(
    default_configuration_name="semantic_config",
    configurations=[
        SemanticConfiguration(
            name="semantic_config",
            prioritized_fields=SemanticPrioritizedFields(
                content_fields=[SemanticField(field_name="page_chunk")]
            ),
        )
    ],
)

index_client.create_or_update_index(
    SearchIndex(
        name=INDEX_NAME,
        fields=fields,
        vector_search=vector_search,
        semantic_search=semantic_search,
    )
)
print(f"Index '{INDEX_NAME}' is ready.")

Index 'earth-at-night-cookbook' is ready.


## Step 3 — Load the NASA *Earth at Night* dataset

The dataset is a pre-chunked JSON file derived from the open NASA e-book. Each row is one page chunk with its embedding already attached, so we can stream them straight into the index with the buffered sender.

In [4]:
import requests
from azure.search.documents import SearchIndexingBufferedSender

DATA_URL = (
    "https://raw.githubusercontent.com/Azure-Samples/azure-search-sample-data"
    "/refs/heads/main/nasa-e-book/earth-at-night-json/documents.json"
)
documents = requests.get(DATA_URL, timeout=60).json()

with SearchIndexingBufferedSender(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential,
) as sender:
    sender.upload_documents(documents=documents)

print(f"Uploaded {len(documents)} documents to '{INDEX_NAME}'.")

Uploaded 194 documents to 'earth-at-night-cookbook'.


## Step 4 — Create the Knowledge Source

A **Knowledge Source** is a thin pointer that tells Foundry IQ *where* a piece of data lives and *which fields* to surface as citations. It does **not** store data itself.

We declare the three fields we want to flow back in every reference:

- `id` for traceability
- `page_chunk` so the LLM sees the underlying text during synthesis
- `page_number` so the answer can cite a real page

In [5]:
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE_NAME,
    description="NASA Earth at Night e-book — chunked pages with vector embeddings.",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        semantic_configuration_name="semantic_config",
        source_data_fields=[
            SearchIndexFieldReference(name="id"),
            SearchIndexFieldReference(name="page_chunk"),
            SearchIndexFieldReference(name="page_number"),
        ],
    ),
)

index_client.create_or_update_knowledge_source(knowledge_source)
print(f"Knowledge source '{KNOWLEDGE_SOURCE_NAME}' is ready.")

Knowledge source 'earth-knowledge-source-cookbook' is ready.


## Step 5 — Create the Knowledge Base

A **Knowledge Base** wraps one or more Knowledge Sources with an **LLM** and a **retrieval strategy**. The choices that matter:

| Setting | What it controls |
|---|---|
| `models` | Chat model used for query planning and answer synthesis |
| `knowledge_sources` | Which sources are in scope for this KB |
| `retrieval_reasoning_effort` | How much the planner reasons (`minimal`, `low`, `medium`) |
| `output_mode` | `extractiveData` (raw chunks) or `answerSynthesis` (cited answer) |
| `answer_instructions` | System prompt that shapes the synthesized answer |

We pick **`answerSynthesis`** so the response is a natural-language answer with citations rather than a chunk dump.

In [6]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
)

gpt_parameters = AzureOpenAIVectorizerParameters(
    resource_url=AOAI_ENDPOINT,
    deployment_name=GPT_DEPLOYMENT,
    api_key=AOAI_API_KEY,
    model_name=GPT_MODEL,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=gpt_parameters)],
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode="answerSynthesis",
    answer_instructions=(
        "Provide a two sentence concise and informative answer "
        "based on the retrieved documents."
    ),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' is ready.")

Knowledge base 'earth-knowledge-base-cookbook' is ready.


## Step 6 — Ask a complex multi-part question

This is where agentic retrieval earns its keep. The query below bundles **two unrelated sub-questions** in one turn — a classic case where naive RAG falls over because a single similarity search cannot satisfy both halves.

The Knowledge Base will:

1. **Plan** — the LLM decomposes the user turn into focused subqueries
2. **Retrieve** — each subquery hits the Search Index in parallel, with the semantic ranker reordering candidates
3. **Synthesize** — the LLM writes one grounded answer that addresses every part

We set `always_query_source=True` so the planner never short-circuits the retrieval, and `include_activity=True` so we can see what it did.

In [7]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)


def make_request(messages: list[dict[str, str]]) -> KnowledgeBaseRetrievalRequest:
    return KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role=m["role"],
                content=[KnowledgeBaseMessageTextContent(text=m["content"])],
            )
            for m in messages
        ],
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
                include_references=True,
                include_reference_source_data=True,
                always_query_source=True,
            )
        ],
        include_activity=True,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    )


def answer_text(result) -> str:
    return "\n\n".join(
        content.text
        for message in (result.response or [])
        for content in (message.content or [])
        if getattr(content, "text", None)
    )


retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint=SEARCH_ENDPOINT,
    credential=credential,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
)

messages = [
    {
        "role": "user",
        "content": (
            "Why do suburban belts display larger December brightening than urban "
            "cores even though absolute light levels are higher downtown? "
            "Why is the Phoenix nighttime street grid so sharply visible from space, "
            "whereas large stretches of the interstate between midwestern cities "
            "remain comparatively dim?"
        ),
    },
]

result = retrieval_client.retrieve(make_request(messages))
first_answer = answer_text(result)

print("Answer\n------")
print(first_answer)

Answer
------
Suburban belts brighten more in December because holiday lighting is concentrated where single-family homes and yards are common (suburbs) so fractional increases there are much larger even though downtown absolute illumination remains higher [ref_id:11][ref_id:3]. The Phoenix street grid appears sharply from space because a regular, dense street-and-surface-lighting pattern (grid streets, commercial nodes, and freeways) produces continuous, bright linear features, whereas long Midwestern interstates often pass through sparsely lit rural areas with few continuous roadside or adjacent urban lights, so the highway appears comparatively dim except near cities [ref_id:0][ref_id:4].


### Inspect the planner activity and references

The `activity` array is the audit trail — it shows every planning, search, reasoning, and synthesis step the Knowledge Base executed, with token counts and latencies. The `references` array is the citation set.

In [8]:
import json

print(f"Activity steps : {len(result.activity or [])}")
print(f"References     : {len(result.references or [])}")
print()
print("Activity trace")
print("--------------")
print(json.dumps([a.as_dict() for a in (result.activity or [])], indent=2))

Activity steps : 5
References     : 19

Activity trace
--------------
[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "inputTokens": 1282,
    "outputTokens": 66,
    "modelName": "gpt-5-mini",
    "elapsedMs": 1701
  },
  {
    "type": "searchIndex",
    "id": 1,
    "knowledgeSourceName": "earth-knowledge-source-cookbook",
    "queryTime": "2026-05-18T22:46:40.8113347Z",
    "count": 5,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "December brightening suburban belts larger than urban cores explanation",
      "filter": null,
      "semanticConfigurationName": null,
      "sourceDataFields": [
        {
          "name": "page_chunk"
        },
        {
          "name": "id"
        },
        {
          "name": "page_number"
        }
      ],
      "searchFields": []
    }
  },
  {
    "type": "searchIndex",
    "id": 2,
    "knowledgeSourceName": "earth-knowledge-source-cookbook",
    "queryTime": "2026-05-18T22:46:41.7063833Z",
    "count": 28,
  

In [9]:
# Show the first few references with their page numbers — these are the citations
# that ground the answer above.
for i, ref in enumerate((result.references or [])[:3]):
    source = ref.source_data or {}
    print(f"[ref_id:{ref.id}] doc_key={ref.doc_key!r}  page={source.get('page_number')}")
    chunk = source.get("page_chunk") or ""
    snippet = chunk[:240].replace("\n", " ")
    if snippet:
        print(f"  {snippet}{'...' if len(chunk) > 240 else ''}")
    print()

[ref_id:0] doc_key='earth_at_night_508_page_104_verbalized'  page=104
  <!-- PageHeader="Urban Structure" -->  ### Location of Phoenix, Arizona  The image depicts a globe highlighting the location of Phoenix, Arizona, in the southwestern United States, marked with a blue pinpoint on the map of North America. Ph...

[ref_id:2] doc_key='earth_at_night_508_page_125_verbalized'  page=125
  # Urban Development  **Date:** October 1, 2013  ---  ## Figure: Urban Development and Infrastructure in the United States  This figure comprises two maps of the continental United States, highlighting the patterns of urban development and i...

[ref_id:4] doc_key='earth_at_night_508_page_124_verbalized'  page=124
  # Urban Development  ## Figure: Location Highlight on Globe  This figure depicts a globe focused on North America, with a marker pinpointing the central region of the United States. The highlighted location represents the geographical focus...



## Step 7 — Continue the conversation (multi-turn)

Agentic retrieval is conversation-aware. We append the assistant's previous answer to the message history and ask a new, narrower question. The planner uses the prior turn as context when deciding what to retrieve next.

In [10]:
messages.append({"role": "assistant", "content": first_answer})
messages.append({"role": "user", "content": "How do I find lava at night?"})

result = retrieval_client.retrieve(make_request(messages))
second_answer = answer_text(result)

print("Follow-up answer\n----------------")
print(second_answer)
print()
print(f"Activity steps : {len(result.activity or [])}")
print(f"References     : {len(result.references or [])}")

Follow-up answer
----------------
To find lava at night, use thermal or nighttime satellite imagery (e.g., VIIRS DNB, thermal infrared bands from Landsat/VIIRS) which detect the hot glow of active lava flows against cooler surroundings, and consult recent satellite passes or eruption alerts from agencies (USGS, local volcano observatories) to know where activity is occurring [ref_id:2][ref_id:0]. For on-the-ground observation prioritize safety and official guidance: check observatory advisories, view from permitted safe vantage points or guided tours, and avoid approaching vents or flows due to extreme heat, gases, and unstable ground (satellite thermal data can help identify active zones before planning a visit) [ref_id:4][ref_id:5].

Activity steps : 5
References     : 17


<!-- foundry-iq-cookbook:mcp-extensions:begin -->
## Step 8 — Talk to the Knowledge Base over MCP (direct JSON-RPC)

Every Foundry IQ Knowledge Base also exposes itself as a **Model Context Protocol** server at:

```
{SEARCH_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version=2026-05-01-preview
```

That means any MCP-capable client — Claude Desktop, VS Code Copilot, a custom Python agent, the Foundry Agent Service, or the Microsoft Agent Framework — can call the *same* retrieval pipeline you built above using a single tool: **`knowledge_base_retrieve`**.

The transport is plain JSON-RPC 2.0 over HTTP with streamable / SSE responses. The cell below:

1. Calls **`tools/list`** to discover the tools the server publishes.
2. Calls **`tools/call`** with `knowledge_base_retrieve` and a natural-language query, then prints the synthesized answer.

The only auth header is the same Search admin `api-key` you've been using all along (for federated Knowledge Sources — SharePoint Remote, Fabric Ontology — you'd also pass an `x-ms-query-source-authorization` user-OBO bearer).

In [11]:
import json
import requests

MCP_URL = f"{SEARCH_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version=2026-05-01-preview"
MCP_HEADERS = {
    "api-key": SEARCH_API_KEY,
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
}


def _parse_mcp_response(response: requests.Response) -> dict:
    """The MCP endpoint may return either JSON or SSE. Coerce both to one dict."""
    content_type = response.headers.get("Content-Type", "")
    if "text/event-stream" in content_type:
        for line in response.text.splitlines():
            if line.startswith("data: "):
                return json.loads(line[len("data: "):])
        raise RuntimeError(f"No data event in SSE response: {response.text[:200]}")
    return response.json()


# 1) Discover tools
list_resp = requests.post(
    MCP_URL,
    headers=MCP_HEADERS,
    json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}},
    timeout=60,
)
list_resp.raise_for_status()
tools_payload = _parse_mcp_response(list_resp)

print("Tools published by the Knowledge Base")
print("-------------------------------------")
for tool in tools_payload["result"]["tools"]:
    print(f"- {tool['name']}: {tool.get('description', '').splitlines()[0]}")
print()

# 2) Invoke knowledge_base_retrieve
call_resp = requests.post(
    MCP_URL,
    headers=MCP_HEADERS,
    json={
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {
            "name": "knowledge_base_retrieve",
            "arguments": {
                "queries": ["What does the dataset show about wildfires visible at night?"]
            },
        },
    },
    timeout=180,
)
call_resp.raise_for_status()
call_payload = _parse_mcp_response(call_resp)

print("knowledge_base_retrieve answer")
print("------------------------------")
for block in call_payload["result"].get("content", []):
    if block.get("type") == "text":
        print(block["text"])


Tools published by the Knowledge Base
-------------------------------------
- knowledge_base_retrieve: Use knowledge_base_retrieve to search for information or documents that must be authoritative and attributable to a source. This knowledge base is always relevant to the user and any organizations they're affiliated with. You may call this tool with ambiguous queries to retrieve relevant context before asking the user for further clarification. If a Knowledge Base Description is provided, use it to understand what content is included/excluded and to scope your retrieval.



knowledge_base_retrieve answer
------------------------------
The dataset (VIIRS DNB and related nightlight products compiled in NASA’s Earth at Night/Black Marble) shows that wildfires are routinely visible from space at night as bright hotspots and illuminated plumes of smoke, and that these night detections can track ignition, spread, peak intensity, and decline over time (e.g., Rim Fire progression Aug–Sep 2013) [ref_id:4][ref_id:0]. It also shows that fires appear as scattered bright points away from cities in many regions (e.g., Australia, Siberia, Pacific Northwest, Mount Vesuvius, Mount Etna) and that nighttime imagery complements daytime thermal/hotspot products for near-real-time monitoring and disaster response [ref_id:16][ref_id:10][ref_id:6][ref_id:14][ref_id:2].


## Step 9 — Wire the Knowledge Base into the Microsoft Agent Framework

The [Microsoft Agent Framework](https://learn.microsoft.com/agent-framework/) ships first-class MCP transports. For Foundry IQ the simplest, most portable pattern is:

| Piece | What it does |
| --- | --- |
| `MCPStreamableHTTPTool` | Connects to the Knowledge Base MCP endpoint and surfaces `knowledge_base_retrieve` as an Agent Framework tool. |
| `OpenAIChatClient` (Azure mode) | Drives the Azure OpenAI **Responses** API, which natively understands tool-calling. |
| `Agent` | The minimal Agent Framework runtime — receives the user message, lets the model decide when to call the tool, and returns the final grounded answer. |

End-to-end the flow is:

1. The user asks a question.
2. The model decides to call `foundry_iq_kb-knowledge_base_retrieve`.
3. The Agent Framework forwards the call to the Knowledge Base over JSON-RPC.
4. Foundry IQ runs query planning → parallel subqueries → semantic ranking → answer synthesis (everything you saw in Step 6) and returns a single grounded blob.
5. The model phrases the final answer with `[ref_id:N]` citations intact.

**Two gotchas worth remembering:**

- The Knowledge Base MCP server is **stateless** — only `tools/list` and `tools/call` are supported. Pass `load_prompts=False` to `MCPStreamableHTTPTool` so the framework doesn't try to call `prompts/list`.
- The `api-key` (or `x-ms-query-source-authorization` for federated KS) must be attached at the *HTTP layer*. Build an `httpx.AsyncClient(headers=...)` and pass it in via `http_client=`.

To swap to a different runtime — `AzureAIAgentClient`, the Foundry Agent Service, a local OpenAI-compatible model — keep the `MCPStreamableHTTPTool` instance and replace the chat client.

In [12]:
import warnings

# The Agent Framework emits ExperimentalWarning on import for the harness/skills
# subsystems — they don't affect this scenario, so we hide them.
warnings.filterwarnings("ignore", message=".*is experimental.*")

import httpx
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework_openai import OpenAIChatClient

mcp_http = httpx.AsyncClient(
    headers={
        "api-key": SEARCH_API_KEY,
        "Accept": "application/json, text/event-stream",
    },
    timeout=httpx.Timeout(120.0, connect=30.0),
)

try:
    async with MCPStreamableHTTPTool(
        name="foundry_iq_kb",
        url=MCP_URL,
        http_client=mcp_http,
        allowed_tools=["knowledge_base_retrieve"],
        approval_mode="never_require",
        load_prompts=False,
    ) as kb_tool:
        chat_client = OpenAIChatClient(
            azure_endpoint=AOAI_ENDPOINT,
            api_key=AOAI_API_KEY,
            api_version="preview",
            model=GPT_DEPLOYMENT,
        )
        agent = Agent(
            client=chat_client,
            name="EarthAtNightAgent",
            instructions=(
                "You are a grounded research assistant. For every user question, "
                "call foundry_iq_kb-knowledge_base_retrieve first and answer "
                "only with information returned by the tool. Always preserve the "
                "[ref_id:N] citations the tool returns."
            ),
            tools=kb_tool,
        )
        response = await agent.run(
            "What patterns of light do auroras create when observed from orbit at night?"
        )
finally:
    await mcp_http.aclose()

print("Agent Framework answer")
print("----------------------")
print(response.text)


Agent Framework answer
----------------------
From orbit, auroras commonly appear in several distinct patterns: broad arcs or bands; wavy curtains and folds (often called curtains); vertical-looking rays or streamers; and rarer coronal or spiral structures [ref_id:1][ref_id:3]. These features arise when charged solar particles excite atmospheric gases and can show green, red, and blue colors depending on altitude and the gas species excited [ref_id:1][ref_id:3]. Satellite and ISS images show auroras as bright, sometimes jagged or swirling lines that can illuminate surface features, extend across hundreds of kilometers, and are detectable by sensors such as the VIIRS Day/Night Band used in NASA’s Earth-at-Night imagery [ref_id:5][ref_id:9].


## Step 10 — Wire the Knowledge Base into a Foundry Agent

The Foundry Agent Service hosts the agent loop, the conversation store, and the MCP runtime for you. To attach a Foundry IQ Knowledge Base you:

1. **Create a `RemoteTool` project connection** that points at the Knowledge Base's MCP URL. Using `authType=ProjectManagedIdentity` lets the Foundry project assume its own identity when calling Search — no per-user tokens stored on the connection.
2. **Create an agent** that declares an `mcp` tool referencing that connection and allow-lists `knowledge_base_retrieve`.
3. **Create a conversation**, then **POST to `/openai/v1/responses`** with the agent reference and your user input. The Responses API turns the chain into a single grounded answer.

The cell below performs all three steps end-to-end, runs one query, and then deletes both the agent and the project connection so this notebook leaves nothing behind.

> **Auth:** uses `DefaultAzureCredential`, so run `az login` first and ensure your account has *Cognitive Services Contributor* (for the agent), *Azure AI User* (for the project endpoint), and write access to the project's resource group (for the connection PUT).

> **Skip:** if `FOUNDRY_PROJECT_ENDPOINT` is not set in your `.env`, this section is skipped gracefully — the rest of the notebook still runs.

In [13]:
import json
import os
import time

import requests
from azure.identity import DefaultAzureCredential

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
FOUNDRY_PROJECT_RESOURCE_ID = os.getenv("FOUNDRY_PROJECT_RESOURCE_ID")
FOUNDRY_AGENT_NAME = os.getenv("FOUNDRY_AGENT_NAME", "foundry-iq-cookbook-agent")
FOUNDRY_CONNECTION_NAME = os.getenv("FOUNDRY_CONNECTION_NAME", "foundry-iq-cookbook-connection")
FOUNDRY_AGENT_MODEL = os.getenv("FOUNDRY_AGENT_MODEL", GPT_DEPLOYMENT)

if not FOUNDRY_PROJECT_ENDPOINT or not FOUNDRY_PROJECT_RESOURCE_ID:
    print(
        "FOUNDRY_PROJECT_ENDPOINT / FOUNDRY_PROJECT_RESOURCE_ID not configured — "
        "skipping the Foundry Agent Service demo.\n"
        "Add them to your .env and re-run this cell to provision the agent."
    )
else:
    credential = DefaultAzureCredential()
    arm_token = credential.get_token("https://management.azure.com/.default").token
    foundry_token = credential.get_token("https://ai.azure.com/.default").token

    arm_headers = {"Authorization": f"Bearer {arm_token}", "Content-Type": "application/json"}
    foundry_headers = {"Authorization": f"Bearer {foundry_token}", "Content-Type": "application/json"}

    # ----- 1. Project connection -> Knowledge Base MCP endpoint -----
    connection_url = (
        f"https://management.azure.com{FOUNDRY_PROJECT_RESOURCE_ID}"
        f"/connections/{FOUNDRY_CONNECTION_NAME}?api-version=2025-10-01-preview"
    )
    connection_body = {
        "name": FOUNDRY_CONNECTION_NAME,
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "ProjectManagedIdentity",
            "category": "RemoteTool",
            "target": MCP_URL,
            "isSharedToAll": True,
            "audience": "https://search.azure.com/",
            "metadata": {"ApiType": "Azure"},
        },
    }
    r = requests.put(connection_url, headers=arm_headers, json=connection_body, timeout=60)
    r.raise_for_status()
    print(f"Project connection ready: {FOUNDRY_CONNECTION_NAME}")

    # ----- 2. Agent with MCP tool referencing the connection -----
    agent_url = f"{FOUNDRY_PROJECT_ENDPOINT}/agents?api-version=v1"
    agent_body = {
        "name": FOUNDRY_AGENT_NAME,
        "definition": {
            "model": FOUNDRY_AGENT_MODEL,
            "instructions": (
                "Always call knowledge_base_retrieve before answering. "
                "Quote the [ref_id:N] citations the tool returns."
            ),
            "tools": [
                {
                    "type": "mcp",
                    "server_label": "knowledge-base",
                    "server_url": MCP_URL,
                    "require_approval": "never",
                    "allowed_tools": ["knowledge_base_retrieve"],
                    "project_connection_id": FOUNDRY_CONNECTION_NAME,
                }
            ],
            "kind": "prompt",
        },
    }
    # POST creates the agent; if it already exists from a prior run, replace it.
    r = requests.post(agent_url, headers=foundry_headers, json=agent_body, timeout=60)
    if r.status_code == 409:
        requests.delete(
            f"{FOUNDRY_PROJECT_ENDPOINT}/agents/{FOUNDRY_AGENT_NAME}?api-version=v1",
            headers=foundry_headers,
            timeout=60,
        )
        time.sleep(1)
        r = requests.post(agent_url, headers=foundry_headers, json=agent_body, timeout=60)
    r.raise_for_status()
    print(f"Agent ready: {FOUNDRY_AGENT_NAME}")

    # ----- 3. Conversation + first turn -----
    r = requests.post(
        f"{FOUNDRY_PROJECT_ENDPOINT}/openai/v1/conversations",
        headers=foundry_headers,
        json={},
        timeout=30,
    )
    r.raise_for_status()
    conversation_id = r.json()["id"]

    r = requests.post(
        f"{FOUNDRY_PROJECT_ENDPOINT}/openai/v1/responses",
        headers=foundry_headers,
        json={
            "conversation": conversation_id,
            "input": "How are city lights used to track urbanization patterns over time?",
            "agent_reference": {"type": "agent_reference", "name": FOUNDRY_AGENT_NAME},
        },
        timeout=180,
    )
    r.raise_for_status()
    response_payload = r.json()

    answer_chunks = []
    for item in response_payload.get("output", []):
        if item.get("type") == "message":
            for c in item.get("content", []):
                if c.get("type") in ("output_text", "text"):
                    answer_chunks.append(c.get("text", ""))
    print("\nFoundry agent answer")
    print("--------------------")
    print("\n".join(answer_chunks) or json.dumps(response_payload, indent=2)[:1500])

    # ----- 4. Tear down the Foundry-side resources -----
    requests.delete(
        f"{FOUNDRY_PROJECT_ENDPOINT}/agents/{FOUNDRY_AGENT_NAME}?api-version=v1",
        headers=foundry_headers,
        timeout=60,
    )
    requests.delete(connection_url, headers=arm_headers, timeout=60)
    print(f"\nDeleted Foundry agent and project connection.")
# <!-- foundry-iq-cookbook:mcp-extensions:end -->


Project connection ready: earth-at-night-foundry-connection-cookbook


Agent ready: earth-at-night-foundry-agent-cookbook



Foundry agent answer
--------------------
City lights are used as a proxy for human settlement and economic activity, so when scientists compare night-time satellite images over multiple years, they can see urbanization expand, intensify, or reorganize.【5:4†source】【5:5†source】

In practice, they look for:
- **Brighter, larger light footprints** around cities, which indicate urban sprawl and infill growth over time.【5:3†source】【5:16†source】
- **New linear light patterns** along highways, corridors, and transport routes, showing infrastructure-driven development.【5:5†source】【5:8†source】【5:15†source】
- **Emerging satellite towns and peri-urban areas**, which appear as new light clusters around older city cores.【5:10†source】【5:6†source】
- **Regional contrasts** in illumination, which reveal differences in electrification, population concentration, and development levels.【5:12†source】【5:4†source】

A time series makes the change especially clear. For example, Las Vegas shows a dramatic expa


Deleted Foundry agent and project connection.


## Step 11 — Clean up

The Knowledge Base, Knowledge Source, and Search Index are all chargeable resources. The cell below deletes them in dependency order (KB → KS → Index) so a re-run of this notebook starts from a blank slate.

Comment this cell out if you want to keep the resources around to wire up the Foundry Agent Service (see *Next steps* below).

In [14]:
from azure.core.exceptions import ResourceNotFoundError

for name, delete_fn in [
    (KNOWLEDGE_BASE_NAME, index_client.delete_knowledge_base),
    (KNOWLEDGE_SOURCE_NAME, index_client.delete_knowledge_source),
    (INDEX_NAME, index_client.delete_index),
]:
    try:
        delete_fn(name)
        print(f"Deleted {name}.")
    except ResourceNotFoundError:
        print(f"Skipped {name} (already gone).")

Deleted earth-knowledge-base-cookbook.


Deleted earth-knowledge-source-cookbook.


Deleted earth-at-night-cookbook.


## Next steps

You now have the full **Index → Knowledge Source → Knowledge Base** pipeline working with agentic retrieval, multi-turn context, and answer synthesis. From here you can:

- **Productionize the Foundry Agent Service path** — Step 10 used `ProjectManagedIdentity`; in production you can also wire **per-user OBO** for federated Knowledge Sources (SharePoint Remote, Fabric Ontology) by passing `x-ms-query-source-authorization` via `structured_inputs`. See: [Connect a knowledge base to a Foundry agent](https://learn.microsoft.com/azure/foundry/agents/how-to/foundry-iq-connect).
- **Add more Knowledge Sources** — the same Knowledge Base can fan out to Blob, OneLake, SharePoint (indexed or remote), and Web sources. Start with the [Knowledge Source overview](https://learn.microsoft.com/azure/search/agentic-knowledge-source-overview).
- **Tune retrieval** — raise `retrieval_reasoning_effort` to `medium` for harder queries, or switch `output_mode` to `extractiveData` if you want raw chunks for a downstream model.
- **Move to managed identity** — swap `AzureKeyCredential` for `DefaultAzureCredential` and assign *Search Service Contributor*, *Search Index Data Contributor*, and *Cognitive Services User* roles. Reference: [RBAC for Azure AI Search](https://learn.microsoft.com/azure/search/search-security-rbac).

### Reference docs

- [Agentic retrieval overview](https://learn.microsoft.com/azure/search/agentic-retrieval-overview)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Answer synthesis](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-answer-synthesis)
- [Migration: 2025-08 → 2025-11 (`knowledgeAgents` → `knowledgeBases`)](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-migrate)
